In [4]:
!pip install rdkit
import os
import json
import random
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
# 4. Teste rápido com moléculas do dataset
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

def carregar_vocabulario(path="vocab_oficial.json"):
    if os.path.exists(path):
        with open(path, "r") as f:
            char_to_idx = json.load(f)
        logger.info(f"Vocabulário carregado: {len(char_to_idx)} tokens")
    else:
        raise FileNotFoundError(f"Vocabulário não encontrado em {path}. Rode o pré-processamento primeiro.")

    if '<PAD>' not in char_to_idx:
        char_to_idx['<PAD>'] = 0
    if '<SOS>' not in char_to_idx:
        char_to_idx['<SOS>'] = len(char_to_idx)
    if '<EOS>' not in char_to_idx:
        char_to_idx['<EOS>'] = len(char_to_idx)

    idx_to_char = {v: k for k, v in char_to_idx.items()}
    return char_to_idx, idx_to_char

def carregar_modelo(path="molgpt_autoencoder_weights.pt",
                    char_to_idx=None):
    """
    Reconstrói a arquitetura do código e carrega os pesos.
    Precisa do char_to_idx para calcular vocab_size.
    """
    checkpoint = torch.load(path, map_location=device, weights_only=True)

    # Usar config salva ou a global
    config = checkpoint.get('config', CONFIG)
    vocab_size = checkpoint.get('vocab_size', len(char_to_idx))

    # Reconstruir arquitetura
    model = build_model(vocab_size, config)

    # Carregar pesos
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    logger.info(f"Pesos carregados de {path} | vocab_size={vocab_size}")
    return model

def inferencia_autoregressiva(model, smiles_input, char_to_idx, idx_to_char, max_len=256):
    """
    Codifica uma SMILES de entrada e gera a reconstrução
    token a token via inferência autoregressiva.
    """
    model.eval()
    sos = char_to_idx['<SOS>']
    eos = char_to_idx['<EOS>']

    # 1. Codificar entrada
    enc_input = [char_to_idx.get(c, 0) for c in smiles_input[:max_len]]
    enc_input = enc_input + [0] * (max_len - len(enc_input))
    enc_input = torch.LongTensor([enc_input]).to(device)

    # 2. Iniciar decoder com SOS
    dec_input = torch.LongTensor([[sos] + [0] * (max_len - 1)]).to(device)

    # 3. Geração autoregressiva
    resultado = []
    with torch.no_grad():
        for t in range(max_len - 1):
            logits = model(enc_input, dec_input)
            next_token = logits[0, t].argmax().item()

            if next_token == eos:
                break
            if next_token == 0:  # PAD
                continue

            resultado.append(idx_to_char.get(next_token, '?'))
            dec_input[0, t + 1] = next_token

    return ''.join(resultado)


In [11]:
# ============================================================
# 2. TRANSFORMERBLOCK (PyTorch) — COM MÁSCARA CAUSAL CORRIGIDA
# ============================================================

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, is_decoder=False):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.rate = rate
        self.is_decoder = is_decoder

        self.att = nn.MultiheadAttention(embed_dim, num_heads, dropout=rate, batch_first=True)

        if self.is_decoder:
            self.cross_att = nn.MultiheadAttention(embed_dim, num_heads, dropout=rate, batch_first=True)
            self.layernorm_cross = nn.LayerNorm(embed_dim, eps=1e-6)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )

        self.layernorm1 = nn.LayerNorm(embed_dim, eps=1e-6)
        self.layernorm2 = nn.LayerNorm(embed_dim, eps=1e-6)
        self.dropout1 = nn.Dropout(rate)
        self.dropout2 = nn.Dropout(rate)

    def forward(self, x, encoder_output=None, enc_key_padding_mask=None, causal_mask=False):
        seq_len = x.size(1)

        # === MÁSCARA CAUSAL ===
        attn_mask = None
        if causal_mask:
            # triangular inferior = 1, superior = 0
            causal = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
            # MultiheadAttention espera mask onde True = "mascarar" (não atender)
            attn_mask = causal == 0  # True onde NÃO pode atender (acima da diagonal)

        # === KEY PADDING MASK (ignorar PAD=0 no self-attention) ===
        # x é (batch, seq_len, embed_dim), tokens PAD têm índice 0
        key_padding_mask = None
        # extrair índices dos tokens do próprio x não é direto,
        # mas podemos passar via parâmetro se necessário

        # Self-attention
        attn_out, _ = self.att(
            x, x, x,
            attn_mask=attn_mask,
            need_weights=False
        )
        attn_out = self.dropout1(attn_out)
        out1 = self.layernorm1(x + attn_out)

        # Cross-attention (decoder only)
        if self.is_decoder and encoder_output is not None:
            cross_out, _ = self.cross_att(
                out1, encoder_output, encoder_output,
                key_padding_mask=enc_key_padding_mask,
                need_weights=False
            )
            out1 = self.layernorm_cross(out1 + cross_out)

        # Feed-forward
        ffn_out = self.ffn(out1)
        ffn_out = self.dropout2(ffn_out)
        return self.layernorm2(out1 + ffn_out)

In [9]:
# ============================================================
# 5. MODELO SEQ2SEQ COMPLETO (PyTorch)
# ============================================================

class Seq2SeqTransformer(nn.Module):
    """
    Transformer Encoder-Decoder para SMILES.
    Equivalente direto ao build_seq2seq_model do Keras.

    Encoder: processa reagentes → representação latente
    Decoder: gera saída token a token com cross-attention no encoder
    """

    def __init__(self, vocab_size, max_len=256, embed_dim=128, ff_dim=512,
                 num_heads=8, num_layers=4, dropout=0.15):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.embed_dim = embed_dim
        self.num_layers = num_layers

        # ==================== ENCODER ====================
        # Embedding de tokens + positional encoding (soma, igual ao Keras)
        self.encoder_emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.encoder_pos_emb = nn.Embedding(max_len, embed_dim)
        self.encoder_dropout = nn.Dropout(dropout)

        # 4 blocos encoder (sem cross-attention, sem máscara causal)
        self.encoder_blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim, rate=dropout, is_decoder=False)
            for _ in range(num_layers)
        ])

        # ==================== DECODER ====================
        self.decoder_emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.decoder_pos_emb = nn.Embedding(max_len, embed_dim)
        self.decoder_dropout = nn.Dropout(dropout)

        # 4 blocos decoder (com cross-attention + máscara causal)
        self.decoder_blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim, rate=dropout, is_decoder=True)
            for _ in range(num_layers)
        ])

        # Camada de saída — retorna LOGITS (sem softmax)
        # F.cross_entropy aplica log_softmax internamente
        self.output_dense = nn.Linear(embed_dim, vocab_size)

    def encode(self, enc_input):
        """
        Encoder: processa reagentes.
        enc_input: (batch, seq_len) — índices dos tokens
        retorna: (batch, seq_len, embed_dim) — representação latente
        """
        batch_size = enc_input.size(0)
        seq_len = enc_input.size(1)

        # Token embeddings
        x = self.encoder_emb(enc_input)  # (batch, seq_len, embed_dim)

        # Positional encoding (soma, igual ao Keras)
        positions = torch.arange(seq_len, device=enc_input.device).unsqueeze(0)  # (1, seq_len)
        x = x + self.encoder_pos_emb(positions)

        x = self.encoder_dropout(x)

        # Passar pelos blocos encoder
        for block in self.encoder_blocks:
            x = block(x, causal_mask=False)

        return x  # encoder_output

    def decode(self, dec_input, encoder_output):
        batch_size = dec_input.size(0)
        seq_len = dec_input.size(1)

        x = self.decoder_emb(dec_input)
        positions = torch.arange(seq_len, device=dec_input.device).unsqueeze(0)
        x = x + self.decoder_pos_emb(positions)
        x = self.decoder_dropout(x)

        # Máscara de PAD do encoder (para cross-attention ignorar PADs)
        enc_padding_mask = (encoder_output.sum(dim=-1) == 0)  # (batch, seq_len) — True onde é PAD

        for block in self.decoder_blocks:
            x = block(x, encoder_output=encoder_output,
                      enc_key_padding_mask=enc_padding_mask, causal_mask=True)

        logits = self.output_dense(x)
        return logits
    def forward(self, enc_input, dec_input):
        """
        Forward completo: encoder → decoder → logits.
        enc_input: (batch, seq_len) — reagentes
        dec_input: (batch, seq_len) — <SOS> + tokens (teacher forcing)
        retorna: (batch, seq_len, vocab_size) — logits
        """
        encoder_output = self.encode(enc_input)
        logits = self.decode(dec_input, encoder_output)
        return logits

def build_model(vocab_size, config):
    """
    Factory function — equivalente ao build_seq2seq_model do Keras.
    """
    model = Seq2SeqTransformer(
        vocab_size=vocab_size,
        max_len=config["max_len"],
        embed_dim=config["embed_dim"],
        ff_dim=config["ff_dim"],
        num_heads=config["num_heads"],
        num_layers=config["num_layers"],
        dropout=config["dropout_rate"],
    )
    model = model.to(device)
    return model

In [13]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Device: {device}")

CONFIG = {
    "vocab_size": 72,
    "max_len": 512,
    "embed_dim": 128,
    "ff_dim": 512,
    "num_heads": 8,
    "num_layers": 4,
    "dropout_rate": 0.15,
    "batch_size": 16,
    "learning_rate": 1e-4,
    "epochs": 3,
}


logger.info("\n" + "=" * 60)
logger.info("TESTE DE RECONSTRUÇÃO")
logger.info("=" * 60)

# Pegar 5 amostras reais do dataset
df_teste = pd.read_csv("/kaggle/input/datasets/glaucosoares/dados-reacoes/dados_reacoes.csv", nrows=50)
# reais = df_teste['input_reagentes'].dropna().astype(str).sample(5, random_state=113).tolist()
reais = df_teste['input_reagentes'].dropna().astype(str).sample(30).tolist()
reais.append("CCO")
reais.append("CC(=O)Oc1ccccc1C(=O)O")
reais.append("CC(=O)Nc1ccc(O)cc1")
reais.append("Cn1cnc2c1c(=O)n(C)c(=O)n2C")
reais.append("CC(C)Cc1ccc(C(C)C(=O)O)cc1")
reais.append("CN1CCCC1c2cccnc2")
reais.append("CN(C)C(=N)N=C(N)N")
reais.append("Clc1cccc2ccccc12")
reais.append("NCCc1ccc(O)c(O)c1")
reais.append("CNCCC(Oc1ccc(C(F)(F)F)cc1)c2ccccc2")

# 5. Recarregar modelo salvo
char_to_idx, idx_to_char = carregar_vocabulario("/kaggle/input/models/glaucosoares/vocab-oficial2/other/default/1/vocab_oficial.json")
model = carregar_modelo("/kaggle/input/models/glaucosoares/molgpt-ep03/pytorch/default/2/molgpt_ep02.pt", char_to_idx=char_to_idx)
model.eval()

for i, smi in enumerate(reais):
    recon = inferencia_autoregressiva(model, smi, char_to_idx, idx_to_char)
    mol_out = Chem.MolFromSmiles(recon)
    valid = "válida" if mol_out else "INVÁLIDA"
    match = "EXATO" if recon == smi else "DIFERENTE"

    # Comparação canônica
    canon = ""
    mol_in = Chem.MolFromSmiles(smi)
    if mol_in and mol_out:
        if Chem.MolToSmiles(mol_in) == Chem.MolToSmiles(mol_out):
            canon = " | MESMA molécula"
        else:
            canon = " | molécula diferente"

    print(f"[{i+1}] Orig:  {smi}")
    print(f"    Rec:   {recon}")
    print(f"    {match} | {valid}{canon}")
    print("-" * 40)

# 5. Para recarregar o modelo depois (exemplo):
# model = carregar_modelo("molgpt_autoencoder_final.pt", char_to_idx=char_to_idx)
# model.eval()
# recon = inferencia_autoregressiva(model, "CCO", char_to_idx, idx_to_char)

2026-09-04 02:34:43,034 - INFO - Device: cpu
2026-09-04 02:34:43,039 - INFO - 
2026-09-04 02:34:43,043 - INFO - TESTE DE RECONSTRUÇÃO
2026-09-04 02:34:43,043 - INFO - ============================================================
2026-09-04 02:34:43,061 - INFO - Vocabulário carregado: 67 tokens
2026-09-04 02:34:43,140 - INFO - Pesos carregados de /kaggle/input/models/glaucosoares/molgpt-ep03/pytorch/default/2/molgpt_ep02.pt | vocab_size=72


[1] Orig:  O=S(=O)(Cl)/C=C/c1ccc(F)cc1.Nc1ccc(Cl)cc1
    Rec:   O=S(=O)(Cl)/C=C/c1ccc(F)cc1.Nc1ccc(Cl)cc1
    EXATO | válida | MESMA molécula
----------------------------------------
[2] Orig:  Cl.CCOC(=O)CCN.COCC1=C(C(=O)OC)[C@H](c2ccc(F)c(F)c2)N(C(=O)Oc2ccc([N+](=O)[O-])cc2)C(=O)N1.CCN(CC)CC.ClCCl
    Rec:   Cl.CCOC(=O)CCN.COCC1=C(C(=O)OC)[C@H](c2ccc(F)c(F)c2)N(C(=O)Oc2ccc([N+](=O)[O-])cc2)C(=O)N1.CCN(CC)CC.ClCCl
    EXATO | válida | MESMA molécula
----------------------------------------
[3] Orig:  N#C[S-].[NH4+].II.Cc1c(N)cccc1Cl.O.CO
    Rec:   N#C[S-].[NH4+].II.Cc1c(N)cccc1Cl.O.CO
    EXATO | válida | MESMA molécula
----------------------------------------
[4] Orig:  CCOC(=O)C(O)Cc1ccc(OCCN2CCOc3ccccc32)cc1.[H-].[Na+].BrCc1ccccc1.O.CN(C)C=O
    Rec:   CCOC(=O)C(O)Cc1ccc(OCCN2CCOc3ccccc32)cc1.[H-].[Na+].BrCc1ccccc1.O.CN(C)C=O
    EXATO | válida | MESMA molécula
----------------------------------------
[5] Orig:  Cl.NCCc1ccc(-c2nc3ccccc3o2)cc1.N#CCCCc1ccc(-c2nc3ccccc3o2)cc1.NCCCCc1